###  **Airbnb NYC Market Analysis — Pricing Drivers & Revenue Opportunities**

 **Project Overview**

 Analyzed 20,000+ Airbnb listings across New York City to identify pricing drivers, room-type demand patterns, host concentration, and availability trends. The project focuses on uncovering actionable insights that could help hosts optimize pricing strategies and understand competitive market dynamics.

 **Key Questions**

 1. Which NYC boroughs provide the highest revenue opportunity for Airbnb hosts?

2. Which room types maximize pricing efficiency and occupancy potential?

1. Which borough has the highest Airbnb prices?

2. How does room type affect pricing?

3. Do cheaper listings receive more reviews?

4. Which hosts dominate the market?

5. How does availability impact engagement?

6. Where are expensive listing concentrated geographically?

**Tools & Technologies**

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Jupyter Notebook

**Dataset Information**

- Source: NYC Airbnb Open Dataset
- Total Records: 20,000+ listings
- Features: Pricing, Room Type, Availability, Reviews, Host Data, Geographic Coordinates

#### **1. Setup & Environment**

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')


print("Libraries Imported Successful")

#### **2. Data Loading**

In [ ]:
df = pd.read_csv('../data/new_york_airbnb_2024.csv', encoding_errors='ignore')

print("Dataset Loaded Successfully")

print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

display(df.head())

#### **3. Initial Data Inspection**


This section evaluates the structure, quality, and completeness of the dataset before performing cleaning and analysis.

The objective is to identify:
- dataset dimensions
- column datatypes
- missing values
- duplicate records
- potential inconsistencies
- overall data reliability

In [ ]:
df.head()

In [ ]:
print("="*20)
print("DATASET OVERVIEW")
print("="*20)

print(f"Total Rows    : {df.shape[0]:,}")
print(f"Total Columns : {df.shape[1]}")

print("\nColumn Names:")
print(df.columns.tolist())

In [ ]:
print("="*10)
print("DATA TYPES")
print("="*10)

display(df.dtypes)

In [ ]:
missing_values = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing_values = missing_values[missing_values > 0]

display(missing_values)

**Observation**
 
 - 7 rows have missing values across multiple columns like - availability, room type, minimum nights etc.
 - 34 rows have missing values in price, will drop during cleaning.
 - baths, bedrooms, rating are stored as str(text) instead of numbers - needs fixing.
 - id and host_id are stored as numbers but they are identifiers, should convert them to object.

#### **4. Data Cleaning & Preparation**

This section focuses on improving dataset quality before analysis.

The cleaning process includes:
- handling missing values
- removing duplicate records
- validating numerical fields
- correcting inconsistent values
- preparing features for analysis

The objective is to ensure analytical accuracy and reliability.

#### **Handling Missing Values** 

In [ ]:
df.isnull().sum().sum()

In [ ]:
missing_values = df.isnull().sum()

missing_percentage = (
    df.isnull().mean() * 100
).round(2)

missing_data = pd.DataFrame({
    'Column Name':missing_values.index,
    'Missing Values':missing_values.values,
    'Percentage': missing_percentage.values
})

display(missing_data[missing_data['Missing Values'] > 0])
                                                    

**Missing Value Observation**

- The dataset contains very few missing values across all columns.
- The highest number of missing values is 34 which is in 'price' column, and the missing percentage is 0.16% while most other columns contain just 0.03% missing values.
- Since the missing data affect less than 1% of the dataset, removing these rows won't affect the overall analysis.

In [ ]:
print(f"Rows before dropping null values: {df.shape[0]:,}")

df=df.dropna()

print(f"Rows after dropping null values: {df.shape[0]:,}")

In [ ]:
df.isnull().sum().sum()

#### **Duplicate handling**

In [ ]:
duplicate_count = df.duplicated().sum()

print(f"Number Of Duplicate Rows: {duplicate_count}")

**Duplicate Observation**

- The dataset contains 12 duplicate rows out of 20736 listing.
- These duplicate records can distort the analysis by inflating counts and affect statistical calculations
- They will be removed before further analysis to ensure accurate and reliable analysis

In [ ]:
duplicate_ids = df[df.duplicated(subset=['id'], keep=False)]
duplicate_ids.sort_values('id')


In [ ]:
df[df.duplicated()]

**Observation**

- These are the duplicate rows 

In [ ]:
print(f"Rows Before Removing Duplicates: {df.shape[0]:,}")

df.drop_duplicates(inplace=True)

print(f"Rows After Removing Duplicates: {df.shape[0]:,}")

df.duplicated().sum()

#### **Datatype Inspection**

In [ ]:
df.dtypes

In [ ]:
print(df['rating'].unique())

In [ ]:
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

print(df['rating'].dtype)

In [ ]:
print(df['bedrooms'].unique())

In [ ]:
df['bedrooms'] = df['bedrooms'].replace('Studio', '0')

df['bedrooms'] = pd.to_numeric(df['bedrooms'])

print(df['bedrooms'].dtype)

df.dtypes

In [ ]:
print(df['baths'].unique())

In [ ]:
df['baths'] = df['baths'].replace('Not specified', '0')

df['baths'] = pd.to_numeric(df['baths'])

print(df['baths'].dtype)

df.dtypes

In [ ]:
print(df['last_review'].unique())

In [ ]:
df['last_review'] = pd.to_datetime(df['last_review'], dayfirst=True)

print(df['last_review'].dtype)

df['last_review'].head()

df.dtypes

**Datatype Conversion**

During datatype inspection, several columns were found to have incorrect datatypes due to mixed or non-numeric values in the dataset.

Datatype conversions were performed in the following columns:

- 'rating' was converted from 'str' to 'float64' using errors=coerce, which automatically converts non-numeric values into 'NaN'

- 'bedrooms' was converted from 'str' to 'int64'
replacing 'studio' with '0' to represent studio apartment without separate bedrooms.

- 'baths' was converted from 'str' to 'float64'
replacing 'Not specified' with '0', to enable numerical analysis.

- 'last_review' was converted from 'str' to 'datetime' using pd.to_datetime() with 'dayfirst=True' because the dates were stored in day/month/year format.


In [ ]:
df.to_csv(
    '../outputs/cleaned_airbnb_data.csv',
    index=False
)

### **5. Exploratory Analysis**
**Outlier Detection & Handling**

Outlier Detection & Handling

- Outliers are extreme values that differ from the majority of the dataset.

- In airbnb listing, 'price' column contains luxury listing with extremely high prices. These values can distort statistical analysis and visualization.

- Therefore, outlier detection is necessary before performing exploratory data analysis.

In [ ]:
plt.figure(figsize=(8, 4))

sns.boxplot(x=df['price'])

plt.title('Price Distribution Before Outlier Removal',
          fontsize=14,
          fontweight='bold')

plt.xlabel('price')

plt.savefig(
    '../images/Price Distribution Before Outlier Removal.png',
    bbox_inches='tight'
)

plt.show()


**IOR Method**

In [ ]:
Q1 = df['price'].quantile(0.25)

Q3 = df['price'].quantile(0.75)

IQR = Q3 - Q1

upper_fence = Q3 +(1.5 * IQR)

print(f"Q1: {Q1}")

print(f"Q3: {Q3}")

print(f"IQR: {IQR}")

print(f"Upper Fence: {upper_fence}")

**IQR Observation**

- The IQR method was used to identify extreme values in the price distribution.

- The IQR method identified prices above $377.5 as potential outliers.

- Listings priced far above the upper_fence is considered as potential outliers because they differ far from the majority of listings and may distort statistical analysis.


In [ ]:
print(f"Listings above $377.5: {(df['price'] >= 377.5).sum()}")

print(f"Listings above $3000: {(df['price'] >= 3000).sum()}")

- Prices above $377.5 is considered as outliers, identified by IQR method.

- But removing these outliers can end up in removing many realistic properties, as this dataset contains premium and luxury listings.

- Therefore, a practical cutoff of $3000 was chosen so it remove only extreme luxury listing while preseving majority of genuine luxury listing.

In [ ]:
print(f"Rows Before Outlier Removal: {df.shape[0]:,}")

df = df[df['price'] < 3000]

print(f"Rows After Outlier Removal: {df.shape[0]:,}")

In [ ]:
plt.figure(figsize=(8, 4))

sns.boxplot(x=df['price'])

plt.title('Price Distribution After Outlier Removal',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Price')

plt.savefig(
    '../images/Price Distribution After Outlier Removal.png',
    bbox_inches='tight'
)

plt.show()

In [ ]:
df['price'].describe()

**Cleaning Observation**

- After removing extreme listing above $3000, the price distribution became more balanced and easier to analyze.

- The dataset still contains some high-priced listings, but it represents realistic permium airbnb properties.

**EXPLORATORY DATA ANALYSIS**

**Univariate Analysis**

It focuses on analyzing one variable at a time to understand its distribution, spread, and overall patterns within the dataset.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df['price'], bins=50, kde=True)

plt.title('Distribution of Airbnb Prices',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Price')

plt.ylabel('Frequency')

plt.savefig(
    '../images/Distribution of Airbnb Prices.png',
    bbox_inches='tight'
)

plt.show()



**Price Distribution Observation**

- The majority of Airbnb listings are concentrated in the lower price range, with most properties priced below $300.

- The distribution is positively skewed (right-skewed), as a small number of high-priced listings extend the tail toward the right side.

- The frequency of listings decreases as price increases, indicating that affordable and mid-range properties dominate the market.

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x='room_type',
    order=df['room_type'].value_counts().index
)

plt.title('Number of Listings by Room Type',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Room Type')

plt.ylabel('Count')

plt.savefig(
    '../images/Number of Listings by Room Type.png',
    bbox_inches='tight'
)

plt.show()

**Room Type Distribution Observation**

- Entire home/apartment listings dominate the Airbnb market, followed by private rooms.

- Shared rooms and hotel rooms represent only a very small proportion of total listings.

- This suggests that most guests prefer private accommodation options rather than shared living spaces.

In [ ]:
plt.figure(figsize=(8, 5))

sns.countplot(
    data=df,
    x='neighbourhood_group',
    order=df['neighbourhood_group'].value_counts().index
)

plt.title('Number of Listings by Borough',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Borough')

plt.ylabel('Count')

plt.savefig(
    '../images/Number of Listings by Borough.png',
    bbox_inches='tight'
)

plt.show()

**Borough Distribution Observation**

- Manhattan contains the highest number of Airbnb listings, closely followed by Brooklyn.

- Queens has a moderate number of listings, while Bronx and Staten Island contribute only a small share of the market.

- This indicates that Airbnb activity is heavily concentrated in Manhattan and Brooklyn, likely due to higher tourism demand, accessibility, and urban activity.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df['number_of_reviews'], bins=50, kde=True)

plt.title('Distribution of Number of Reviews',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Number of Reviews')

plt.ylabel('Frequency')

plt.savefig(
    '../images/Distribution of Number of Reviews.png',
    bbox_inches='tight'
)

plt.show()

**Review Distribution Observation**

- Most Airbnb listings have relatively low review counts, while only a small number of listings receive extremely high numbers of reviews.

- The distribution is heavily right-skewed, indicating that guest engagement is concentrated among a limited number of highly active listings.

- The frequency decreases rapidly as the number of reviews increases, suggesting that only a few properties dominate booking activity and customer interactions.

In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(df['availability_365'], bins=40, kde=True)

plt.title('Distribution of Availability (365 Days)',
         fontsize=14,
         fontweight='bold' )

plt.xlabel('Availability in Days')

plt.ylabel('Frequency')

plt.savefig(
    '../images/Distribution of Availability (365 Days).png',
    bbox_inches='tight'
)

plt.show()

**Availability Distribution Observation**

- A large number of Airbnb listings are available for most of the year, particularly around 365 days.

- Another noticeable group of listings has very low availability, indicating that some properties may either be frequently booked or only occasionally listed.

- The distribution contains several spikes across different availability ranges, suggesting varying hosting strategies and seasonal availability patterns among hosts.

**Bivariate Analysis**

Bivariate analysis examines the relationship between two variables to identify patterns, trends, and possible correlations within the dataset.

In [ ]:
plt.figure(figsize=(8, 5))

sns.boxplot(
    data=df,
    x='room_type',
    y='price'
)

plt.title('Price Distribution by Room Type',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Room Type')

plt.ylabel('Price')

plt.savefig(
    '../images/Price Distribution by Room Type.png',
    bbox_inches='tight'
)

plt.show()

**Price vs Room Type Observation**

- Hotel rooms have the highest median prices among all room types, indicating that they are generally the most expensive accommodation option.

- Entire home/apartment listings also show relatively high prices and large variability, reflecting differences in property size, location, and amenities.

- Private rooms and shared rooms are comparatively more affordable, with shared rooms showing the lowest overall pricing.

- A large number of outliers exist across all room types, suggesting the presence of premium and luxury listings within each category.

In [ ]:
plt.figure(figsize=(9, 5))

sns.boxplot(
    data=df,
    x='neighbourhood_group',
    y='price'
)

plt.title('Price Distribution by Borough',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Borough')

plt.ylabel('Price')

plt.savefig(
    '../images/Price Distribution by Borough.png',
    bbox_inches='tight'
)

plt.show()

**Price Distribution by Borough Observation**

- Manhattan has the highest median price among all boroughs, indicating that it is generally the most expensive Airbnb market in New York.

- Brooklyn also shows relatively high pricing, while Queens, Bronx, and Staten Island have lower median prices.

- Manhattan displays the largest spread and highest number of outliers, suggesting a wide variation in property pricing and the presence of many premium and luxury listings.

- The boroughs with lower median prices also show comparatively smaller price variability.

In [ ]:
plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df,
    x='price',
    y='number_of_reviews',
    alpha=0.5
)

plt.title('Price vs Number of Reviews',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Price')

plt.ylabel('Number of Reviews')

plt.savefig(
    '../images/Price vs Number of Reviews.png',
    bbox_inches='tight'
)

plt.show()

**Price vs Number of Reviews Observation**

- Most Airbnb listings are concentrated in the lower price range with relatively fewer reviews overall.

- Listings with lower and moderate prices tend to receive higher numbers of reviews, suggesting greater customer engagement and booking activity.

- As price increases, the number of reviews generally decreases, indicating a weak negative relationship between pricing and review activity.


**Correlation Analysis**

In [ ]:
corr_matrix = df.corr(numeric_only=True)

corr_matrix

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Correlation Heatmap',
          fontsize=14,
          fontweight='bold')

plt.savefig(
    '../images/Correlation Heatmap.png',
    bbox_inches='tight'
)

plt.show()

**Correlation Analysis Observation**

- The strongest positive correlation is observed between 'reviews_per_month' and 'number_of_reviews_ltm' (0.85), indicating that listings receiving recent reviews also tend to maintain high monthly review activity.

- 'number_of_reviews' and 'reviews_per_month' also show a strong positive relationship, suggesting consistent customer engagement over time.

- 'bedrooms' and 'beds' exhibit a strong positive correlation, which is expected since larger properties generally accommodate more beds.

- Price shows weak-to-moderate positive correlations with 'bedrooms', 'beds', and 'baths', indicating that larger properties tend to be more expensive.

- Most other correlations are relatively weak, suggesting that Airbnb pricing and customer behavior are influenced by multiple interacting factors rather than a single variable.

#### **Advanced Exploratory Data Analysis**

This section focuses on deeper business insights using aggregation, grouping, and feature-based analysis to better understand Airbnb pricing, host activity, and customer behavior.

**Average Price By Room Type**

In [ ]:
avg_price_room = (
    df.groupby('room_type')['price']
    .mean()
    .sort_values(ascending=False)
)

avg_price_room

In [ ]:
plt.figure(figsize=(8, 5))

avg_price_room.plot(kind='bar')

plt.title('Average Price by Room Type',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Room Type')

plt.ylabel('Average Price')

plt.xticks(rotation=0)

plt.savefig(
    '../images/Average Price by Room Type.png',
    bbox_inches='tight'
)

plt.show()

**TOP 10 MOST EXPENSIVE NEIGHBOURBOODS**

In [ ]:
top_neighbourhoods = (
    df.groupby('neighbourhood')['price']
    .mean()
    .sort_values(ascending=False)
    .head(10)
)

top_neighbourhoods

In [ ]:
plt.figure(figsize=(10, 5))

top_neighbourhoods.plot(kind='bar')

plt.title('Top 10 Most Expensive Neighbourhoods',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Neighbourhood')

plt.ylabel('Average Price')

plt.xticks(rotation=45)

plt.savefig(
    '../images/Top 10 Most Expensive Neighbourhoods.png',
    bbox_inches='tight'
)

plt.show()

**Top Neighbourhood Observation**

- Several neighbourhoods show significantly higher average Airbnb prices compared to others.

- Premium neighbourhoods are likely influenced by factors such as tourism demand, luxury properties, accessibility, and central locations.

- This analysis highlights strong location-based pricing differences across New York City.

**CHEAPEST BOROUGHS**

In [ ]:
avg_price_borough = (
    df.groupby('neighbourhood_group')['price']
    .mean()
    .sort_values()
)

avg_price_borough

In [ ]:
plt.figure(figsize=(8,5))

avg_price_borough.plot(kind='bar')

plt.title('Average Price by Borough',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Borough')

plt.ylabel('Average Price')

plt.xticks(rotation=0)

plt.savefig(
    '../images/Average Price by Borough.png',
    bbox_inches='tight'
)

plt.show()

In [ ]:
borough_price = (
    df.groupby('neighbourhood_group')['price']
      .median()
      .sort_values(ascending=False)
)

borough_price.to_csv(
    '../outputs/borough_price_summary.csv'
)

**Borough Price Observation**

- Manhattan has the highest average Airbnb prices among all boroughs.

- Brooklyn also maintains relatively high pricing compared to other boroughs.

- Bronx and Staten Island appear to be the most affordable boroughs.

- The results indicate strong geographical influence on Airbnb pricing patterns.

**TOP HOSTS ANALYSIS**

In [ ]:
top_hosts = (
    df.groupby('host_name')['calculated_host_listings_count']
    .max()
    .sort_values(ascending=False)
    .head(10)
)

top_hosts

In [ ]:
plt.figure(figsize=(10,5))

top_hosts.plot(kind='bar')

plt.title('Top Hosts by Listing Count',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Host Name')

plt.ylabel('Number of Listings')

plt.xticks(rotation=45)

plt.savefig(
    '../images/Top Hosts by Listing Count.png',
    bbox_inches='tight'
)

plt.show()

**Host Analysis Observation**

- A small number of hosts manage a significantly large number of listings.

- This suggests the presence of professional hosts or commercial property operators within the Airbnb market.

- Most hosts likely manage only a limited number of properties.

**TIME BASE-REVIEW ANALYSIS**

In [ ]:
reviews_by_year = (
    df['last_review']
    .dt.year
    .value_counts()
    .sort_index()
)

reviews_by_year

In [ ]:
plt.figure(figsize=(10,5))

reviews_by_year.plot(kind='line', marker='o')

plt.title('Review Activity by Year',
          fontsize=14,
          fontweight='bold')

plt.xlabel('Year')

plt.ylabel('Number of Reviews')

plt.savefig(
    '../images/Review Activity by Year.png',
    bbox_inches='tight'
)

plt.show()

**Review Trend Observation**

- Review activity varies across different years, reflecting changes in Airbnb usage and customer engagement over time.

- Certain years show significantly higher review counts, indicating periods of increased platform activity and travel demand.

**FEATURE ENGINEERING SECTION**

Feature engineering involves creating new variables from existing data to improve analysis and generate deeper business insights.

In [ ]:
df['price_per_bedroom'] = df['price'] / (df['bedrooms'] + 1)

**Price Per Bedroom Feature**

This feature estimates pricing efficiency relative to property size by comparing listing price with number of bedrooms.

In [ ]:
df['high_availability'] = df['availability_365'] > 300

**High Availability Feature**

This feature identifies listings available for most of the year, which may represent commercial or highly active rental properties.

In [ ]:
df.to_csv(
    '../outputs/airbnb_feature_engineered.csv',
    index=False
)

### **Pricing Intelligence Analysis**

This section analyzes Airbnb pricing behavior across New York City to identify pricing distribution patterns, premium market segments, borough-level pricing differences, and room-type pricing behavior.

The objective is to understand how pricing varies across the NYC Airbnb ecosystem and identify potential market opportunities.

**PRICE DISTRIBUTION ANALYSIS**

In [ ]:
price_df = df[df['price'] < 1000]

In [ ]:
plt.figure(figsize=(12,6))

sns.histplot(
    price_df['price'],
    bins=50,
    kde=True
)

plt.title(
    "Most NYC Airbnb Listings Are Concentrated Below $300 Per Night",
    fontsize=14,
    fontweight='bold'
)

plt.xlabel("Price Per Night")
plt.ylabel("Number of Listings")

plt.savefig(
    '../images/Most NYC Airbnb Listings Are Concentrated Below $300 Per Night.png',
    bbox_inches='tight'
)

plt.show()

**BOROUGH-LEVEL PRICING ANALYSIS**

In [ ]:
borough_price = (
    df.groupby('neighbourhood_group')['price']
      .median()
      .sort_values(ascending=False)
)

display(borough_price)

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=price_df,
    x='neighbourhood_group',
    y='price'
)

plt.title(
    "Manhattan Maintains The Highest Airbnb Pricing Levels",
    fontsize=14,
    fontweight='bold'
)

plt.xlabel("Borough")
plt.ylabel("Price")

plt.ylim(0,1000)

plt.savefig(
    '../images/Manhattan Maintains The Highest Airbnb Pricing Levels.png',
    bbox_inches='tight'
)

plt.show()

**Insight**

Manhattan maintains the highest median Airbnb pricing levels, reinforcing its premium market positioning within NYC.

Brooklyn supports comparatively moderate pricing while still demonstrating strong market activity, suggesting balanced demand across mid-range segments.

**ROOM-TYPE PRICING ANALYSIS**

In [ ]:
room_price = (
    df.groupby('room_type')['price']
      .median()
      .sort_values(ascending=False)
)

display(room_price)

In [ ]:
plt.figure(figsize=(10,5))

sns.boxplot(
    data=price_df,
    x='room_type',
    y='price'
)

plt.title(
    "Entire Homes Command Premium Airbnb Pricing",
    fontsize=14,
    fontweight='bold'
)

plt.xlabel("Room Type")
plt.ylabel("Price")

plt.ylim(0,1000)

plt.savefig(
    '../images/Entire Homes Command Premium Airbnb Pricing.png',
    bbox_inches='tight'
)

plt.show()

**LUXURY MARKET SEGMENTATION**

In [ ]:
df['price_category'] = pd.cut(
    df['price'],
    bins=[0,100,300,600,10000],
    labels=['Budget','Standard','Premium','Luxury']
)

In [ ]:
price_segment = (
    df['price_category']
      .value_counts()
)

display(price_segment)

In [ ]:
plt.figure(figsize=(8,5))

sns.countplot(
    data=df,
    x='price_category',
    order=['Budget','Standard','Premium','Luxury']
)

plt.title(
    "Most NYC Listings Fall Within Budget & Standard Segments",
    fontsize=14,
    fontweight='bold'
)

plt.xlabel("Price Segment")
plt.ylabel("Listing Count")

plt.savefig(
    '../images/Most NYC Listings Fall Within Budget & Standard Segments.png',
    bbox_inches='tight'
)

plt.show()

**Insight**

The NYC Airbnb ecosystem is heavily concentrated within budget and standard pricing tiers, while premium and luxury listings represent a smaller but high-value segment of the market.

#### **CONCLUSION**

This exploratory data analysis project examined Airbnb listings across New York City to understand pricing patterns, customer engagement, property availability, and host activity.

Key findings from the analysis include:

- Airbnb listings are heavily concentrated in Manhattan and Brooklyn.

- Entire home/apartment listings dominate the market, while hotel and shared room listings are comparatively limited.

- Manhattan has the highest pricing levels and the greatest price variability, indicating the presence of both standard and luxury listings.

- Airbnb prices show a heavily right-skewed distribution, with most listings concentrated in lower-to-mid price ranges.

- Lower and moderately priced listings tend to receive higher numbers of reviews, suggesting stronger customer engagement.

- Correlation analysis revealed strong relationships between review-related variables and moderate relationships between property size features and pricing.

- Host analysis suggests that a small number of professional hosts manage a large proportion of listings.

Overall, the project demonstrates how exploratory data analysis can uncover valuable business insights, market behavior, and customer trends within the Airbnb marketplace.